# 1 - Setup

In [ ]:
%%capture
!pip install -q "transformers==4.57.1" accelerate genomic-benchmarks

# 2 - Model & Tokenizer download

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

MODEL_NAME = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species" #small model that can run on Colab on CPU only resources

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
).to("cpu").eval()

In [ ]:
sequence = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"

In [ ]:
tokens = tokenizer.tokenize(sequence) # Inspect how the sequence is split into tokens

print("Original sequence:")
print(sequence)

print("\nTokens:")
print(tokens)

In [ ]:
inputs = tokenizer(
    sequence,
    return_tensors="pt"
)    # Convert the sequence into model-ready tensors

In [ ]:
import torch

with torch.inference_mode():
    outputs = model(
        **inputs,
        output_hidden_states=True
    )

# Take the embedding produced for each token by the final transformer layer
token_embeddings = outputs.hidden_states[-1]

print("Number of sequences:", token_embeddings.shape[0])
print("Number of tokens:", token_embeddings.shape[1])
print("Embedding size per token:", token_embeddings.shape[2])

In [ ]:
token_index = 0 # select the first token

embedding = token_embeddings[0, token_index] # [0] selects the first (and only) sequence

print("Token:", tokens[token_index])

print("\nEmbedding shape:", embedding.shape) # number of dimensions in the token embedding

print("\nEmbedding values:")
print(embedding[:10])    # print first 10
print("...")
print(embedding[-10:])   # print last 10

# 3 - Use a fine-tuned foundation model to predict enhancers from DNA sequences

In [ ]:
import pandas as pd

URL = (
    "https://zenodo.org/records/15324459/files/human_enhancers_cohn_test.csv.gz"
)    # URL to the test dataset of human enhancer sequences

test_df = pd.read_csv(URL)  # store the test dataset in a dataframe

In [ ]:
test_df.head(10)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = (
    "stephaniesamm/nucleotide-transformer-v2-50m-multi-species-finetuned-human-enhancers-cohn"
)   # huggingface model id of the fine-tuned classifier

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

classifier = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
).to("cuda").eval()  # load the fine-tuned classifier

In [ ]:
import torch

sequences = test_df["sequence"].tolist()

batch_size = 32  # number of sequences to process in each batch

all_predictions = []
all_probabilities = []

for start in range(0, len(sequences), batch_size):
    batch_sequences = sequences[start:start + batch_size]

    # Tokenize the batch and move the resulting tensors to the GPU
    inputs = tokenizer(
        batch_sequences,
        return_tensors="pt"
    ).to("cuda")

    # Run the classifier on this batch
    with torch.inference_mode():
        outputs = classifier(**inputs)

    # Convert the model's logits into class probabilities
    probabilities = torch.softmax(outputs.logits, dim=-1)

    # Select the class with the highest probability for each sequence
    predictions = probabilities.argmax(dim=-1)

    all_predictions.extend(predictions.cpu().tolist())
    all_probabilities.extend(probabilities.cpu().tolist())

In [ ]:
test_df["prediction"] = all_predictions  # add the predictions to the dataframe

test_df["confidence"] = [
    max(probs)
    for probs in all_probabilities
]  # add the confidence scores to the dataframe

test_df["probability_enhancer"] = [
    probs[1]
    for probs in all_probabilities
] # add the probability of the enhancer class to the dataframe

test_df.head()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(
    test_df["label"],
    test_df["prediction"]
)

print(f"Accuracy: {accuracy:.4f}")
print(classification_report(test_df["label"], test_df["prediction"]))